# DDI GNN training on a free GPU (Colab or Kaggle)

Clones the repo, rebuilds the cleaned data and graph cache from the raw DrugBank file,
checks they match the committed `results/cleaning_log.json`, then runs `src/train.py`.

**Before running**
- **Colab:** Runtime → Change runtime type → **T4 GPU**. Checkpoints and logs go to Google Drive
  (`MyDrive/ddi_project_runs`), so a disconnect loses at most one epoch: just re-run all cells.
- **Kaggle:** Settings → Accelerator **GPU T4**, and turn **Internet on**. Outputs go to
  `/kaggle/working/ddi_runs`. For long runs use *Save Version → Save & Run All* so it keeps going
  after you close the tab. To continue runs started elsewhere, upload that output folder (e.g. the
  Colab `ddi_project_runs` Drive folder, as a zip) as a Kaggle dataset and add it as an input.
- Set `REPO_URL` below. For a private repo, add a secret named `GITHUB_TOKEN` (Colab: key icon in
  the sidebar; Kaggle: Add-ons → Secrets) holding a GitHub token with read access.

Every run uses `--resume`: finished runs are skipped, interrupted ones continue from their last epoch.

In [ ]:
REPO_URL = "https://github.com/KritiWadhawan/ddi-project.git"
BRANCH = "master"

# One entry per training run, in order. Arguments go straight to src/train.py.
#
# Round 1 (current): S1 screening of class weighting (balanced = what the RF baseline
# uses, sqrt = gentler) x regularization. The plain-loss runs (no flags) already finished
# for seeds 0-4. Seed 0 gets the full 2x2; balanced was unstable and 2-4x slower to
# converge, so seeds 1-2 run only the sqrt variants. Pick the winner by mean
# *validation* macro-F1, never test.
REG = "--gnn-dropout 0.3 --adamw --weight-decay 1e-2 --lr 5e-4"
VARIANTS = [
    "--class-weights balanced",
    "--class-weights sqrt",
    f"--class-weights balanced {REG}",
    f"--class-weights sqrt {REG}",
]
RUNS = [f"--split S1 --seed 0 {v}" for v in VARIANTS] + [
    f"--split S1 --seed {s} {v}" for s in (1, 2) for v in VARIANTS[1::2]
]

# Round 2 (after picking BEST from round 1): uncomment and run.
# BEST = "--class-weights sqrt"  # <- the winning variant's flags
# RUNS = (
#     [f"--split S1 --seed {s} {BEST}" for s in (3, 4)]
#     + [f"--split S2 --seed {s} {BEST}" for s in range(5)]
#     + [f"--split transductive --seed 0 {BEST}"]
#     + [f"--split S1 --seed 0 {BEST} --readout mean",   # ablation: co-attention -> mean pooling
#        f"--split S1 --seed 0 {BEST} --no-inter"]       # ablation: drop the bipartite GATConv
# )
COMMON_ARGS = "--batch-size 256 --workers 2 --resume"

## Environment, GPU check, persistent output folder

In [ ]:
import os, sys, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
print("Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "other")
!nvidia-smi --query-gpu=name,memory.total --format=csv

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/ddi_project_runs")
elif ON_KAGGLE:
    OUT = Path("/kaggle/working/ddi_runs")
else:
    OUT = Path.home() / "ddi_project_runs"
OUT.mkdir(parents=True, exist_ok=True)
if ON_KAGGLE and not (OUT / "gnn_metrics.csv").exists():
    # Carry over earlier runs (e.g. the Colab Drive folder uploaded as a Kaggle dataset)
    # so finished runs are skipped and interrupted ones resume.
    import shutil
    for prev in Path("/kaggle/input").rglob("gnn_metrics.csv"):
        shutil.copytree(prev.parent, OUT, dirs_exist_ok=True)
        print("carried over earlier runs from", prev.parent)
        break
REPO = Path("/tmp/ddi-project")
print("outputs ->", OUT)

## Clone the repo

In [ ]:
def secret(name):
    try:
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, url, str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1

## Install dependencies

Colab and Kaggle already have CUDA PyTorch, pandas, pyarrow and scikit-learn. Only PyG and RDKit
are added, pinned to the versions used locally so featurization matches.

In [ ]:
!pip install -q torch_geometric==2.8.1 rdkit==2026.3.6
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: switch the runtime/accelerator to a GPU first"

## Link outputs to the persistent folder

`checkpoints/`, `results/runs/` and `results/gnn_metrics.csv` live in `OUT`, so they survive a
disconnect and `--resume` finds them.

In [ ]:
(OUT / "checkpoints").mkdir(exist_ok=True)
(OUT / "runs").mkdir(exist_ok=True)
for link, target in [
    (REPO / "checkpoints", OUT / "checkpoints"),
    (REPO / "results" / "runs", OUT / "runs"),
    (REPO / "results" / "gnn_metrics.csv", OUT / "gnn_metrics.csv"),
]:
    if not link.is_symlink():
        link.symlink_to(target)
!ls -la checkpoints results/runs results/gnn_metrics.csv

## Rebuild the data

Downloads the raw DrugBank DDI file (cached in `OUT`), then runs the same scripts as locally.
The cleaned counts must match the committed `results/cleaning_log.json`.

In [ ]:
import json, shutil

expected = json.loads(Path("results/cleaning_log.json").read_text())

raw_cache = OUT / "drugbank_ddi.tab"
if not raw_cache.exists():
    !wget -q -O {raw_cache} https://dataverse.harvard.edu/api/access/datafile/4139573
Path("data").mkdir(exist_ok=True)
shutil.copy(raw_cache, "data/drugbank_ddi.tab")

!python src/convert_to_parquet.py
!python src/clean.py > /dev/null
!python src/graphs.py

got = json.loads(Path("results/cleaning_log.json").read_text())
for key in ["raw_rows", "conflicting_pairs", "clean_rows", "clean_drugs", "clean_classes"]:
    assert got[key] == expected[key], f"{key}: got {got[key]}, committed {expected[key]}"
print("cleaned data matches the committed cleaning log")

## Speed check

Times a few training steps of the full model without writing anything, and projects the epoch time.

In [ ]:
import time
import torch.nn.functional as F
sys.path.insert(0, str(REPO / "src"))
from graphs import load_graphs
from splits import load_data, make_split
from model import DDIModel
from train import PairLoader

train_df, _, _ = make_split("S1", load_data(), 0)
loader = PairLoader(train_df, load_graphs(), 256, shuffle=True)
model = DDIModel().cuda()
opt = torch.optim.Adam(model.parameters())
times = []
for i, (a, b, y) in zip(range(6), loader):
    t0 = time.time()
    loss = F.cross_entropy(model(a.cuda(), b.cuda()), y.cuda())
    opt.zero_grad(); loss.backward(); opt.step()
    torch.cuda.synchronize()
    times.append(time.time() - t0)
per_batch = sum(times[1:]) / len(times[1:])
print(f"{per_batch:.2f} s/batch (GPU step only) -> ~{per_batch * len(loader) / 60:.1f} min per S1 epoch + validation")
print(f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
del model, opt
torch.cuda.empty_cache()

## Train

Runs everything in `RUNS` in order. Safe to re-run this cell after a disconnect: finished runs are
skipped and interrupted ones resume. The stop button ends the whole loop, not just the current run.

In [ ]:
import shlex

for run_args in RUNS:
    print("=" * 80, f"\n{run_args}\n" + "=" * 80, flush=True)
    cmd = [sys.executable, "-u", "-W", "ignore", "src/train.py", *shlex.split(run_args), *shlex.split(COMMON_ARGS)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("\nStopped. Re-run this cell later to resume where it left off.")
        break
    if proc.wait() != 0:
        raise RuntimeError(f"train.py exited with code {proc.returncode} for: {run_args}")

## Results so far (vs. RF+Morgan baseline)

In [ ]:
import pandas as pd

gnn = pd.read_csv("results/gnn_metrics.csv")
gnn["variant"] = gnn["model"].str.replace(r"^(S1|S2|transductive)_seed\d+_", "", regex=True)
summary = (gnn.groupby(["split", "variant", "set"])[["acc", "macro_f1"]]
              .agg(["mean", "std", "count"]).round(3))
display(summary)

# Round 1 selection: S1 seeds 0-2 only, so every variant is compared on the same seeds.
s1 = gnn[(gnn.split == "S1") & gnn.seed.isin([0, 1, 2]) & (gnn.set == "val")]
pick = s1.groupby("variant").macro_f1.agg(["mean", "std", "count"]).round(3)
display(pick.sort_values("mean", ascending=False).rename(columns={"mean": "val_macro_f1_mean"}))

base = pd.read_csv("results/baseline_summary.csv")
display(base[base.model == "RF+Morgan"][["split", "set", "n_seeds", "acc_mean", "macro_f1_mean", "macro_f1_std"]])

## Bring results back to the local repo

Download this zip and unpack it into the local repo's `results/` folder to commit the logs and
metrics. Checkpoints stay in `OUT` (they are gitignored); copy them separately for Phase 4.

In [ ]:
!cd {OUT} && rm -f gnn_results.zip && zip -qr gnn_results.zip runs gnn_metrics.csv && ls -la gnn_results.zip
if ON_COLAB:
    from google.colab import files
    files.download(str(OUT / "gnn_results.zip"))